# Training convergence — 260723 vs 260914

Asks whether each batch had finished learning at the checkpoints being compared, by reading the
training curves rather than the behavioural sims.

**Why this matters.** The two batches were compared at a matched 4e6 steps, on the assumption that
matching the step count matches how far training had got. It does not, because the schedules are
tied to `max_steps` and the two batches set it differently:

| setting | 260723 | 260914 |
| --- | --- | --- |
| `learning_rate_schedule` | linear | constant |
| `epsilon_schedule` | linear | constant |
| `max_steps` | 4e6 | 7e6 |
| extrinsic gamma | 0.99 | 0.998 |
| lambd | 0.95 | 0.99 |
| `time_horizon` | 128 | 512 |

A linear schedule decays the learning rate to zero **at `max_steps`**. So at 4e6 the 260723 runs are
at the end of their anneal with a learning rate near zero, while the 260914 runs are 4/7 of the way
through at a constant full learning rate. Matching the step count compares a finished, annealed model
against an unfinished, unannealed one.

Three signals are read per run. `Policy/Learning Rate` shows where each run sits in its schedule.
`Self-play/ELO` is the standard self-play progress measure, rising while an agent still beats its
earlier selves. `Environment/Episode Length` falls as trials are completed faster, which is the
training-side counterpart of the `latency_to_reach_wall` and `path_ratio` measured in the sims.

Run in the `mlagents` env, which is the one with tensorboard. Reads event files only; nothing is
trained or written.

In [ ]:
from pathlib import Path

import numpy as np
from tensorboard.backend.event_processing import event_accumulator

RESULTS = Path("~/Unity/Octagon/results").expanduser()
BEH = "OctagonAgentSocial"

# 260723 central models and the 260914 central models simulated at 4e6, so the curves line up
# with the behavioural comparison
RUNS = {
    "260723": [f"260723_{i:04d}" for i in (34, 39, 30, 55, 11, 9)],
    "260914": [f"260914_{i:04d}" for i in (12, 23, 20, 7, 21, 5, 13, 14)],
}
BATCH_DIR = {"260723": RESULTS / "2607" / "260723", "260914": RESULTS / "2609" / "260914"}


def series(run_dir, tag):
    """ Return (steps, values) for one scalar tag, merged over a run's event files.
        Takes a run's behaviour dir and a tensorboard tag.

        A resumed run writes a second event file, so later files overwrite earlier ones on a
        step collision and the merged series stays monotonic. """

    pts = {}
    for f in sorted(Path(run_dir).glob("events.out.tfevents*")):
        ea = event_accumulator.EventAccumulator(str(f), size_guidance={"scalars": 0})
        try:
            ea.Reload()
        except Exception:
            continue
        if tag not in ea.Tags()["scalars"]:
            continue
        for e in ea.Scalars(tag):
            pts[e.step] = e.value
    steps = np.array(sorted(pts))
    return steps, np.array([pts[k] for k in steps])


def slope_per_1e6(steps, values, lo, hi):
    """ Return the least-squares change per 1e6 steps over a window. Takes a series and its bounds. """

    m = (steps >= lo) & (steps <= hi)
    if m.sum() < 5:
        return np.nan
    return np.polyfit(steps[m], values[m], 1)[0] * 1e6


curves = {(b, r): {} for b, rs in RUNS.items() for r in rs}
for b, rs in RUNS.items():
    for r in rs:
        d = BATCH_DIR[b] / r / BEH
        for tag in ("Policy/Learning Rate", "Self-play/ELO", "Environment/Episode Length"):
            curves[(b, r)][tag] = series(d, tag)
print(f"loaded curves for {len(curves)} runs")

## Where each run sits in its schedule

The learning rate says directly whether a run was still being allowed to learn.

In [ ]:
# print the learning rate at a few points, so the shape of each schedule is visible
MARKS = [2e6, 3e6, 3.5e6, 4e6, 5e6, 6e6, 7e6]
print(f"{'run':<14}" + "".join(f"{m/1e6:>11.1f}e6" for m in MARKS))
for (b, r), c in curves.items():
    s, v = c["Policy/Learning Rate"]
    cells_out = []
    for m in MARKS:
        i = np.argmin(np.abs(s - m))
        cells_out.append(f"{v[i]:>13.2e}" if abs(s[i] - m) < 1e5 else f"{'-':>13}")
    print(f"{r:<14}" + "".join(cells_out))

260723 decays to effectively zero at 3e6, which was its original `max_steps`. The extension to 4e6
restarted a second, much smaller decay that again ends near zero. 260914 holds its full learning rate
throughout, and is never annealed at all.

So at the matched 4e6 checkpoint the two batches are in different training regimes, not at the same
point in the same process.

## Was learning still happening?

ELO and episode length, averaged over windows. ELO rising means the policy still beats its earlier
selves; episode length falling means trials are still being completed faster.

In [ ]:
WINDOWS = [(1e6, 2e6), (2e6, 3e6), (3e6, 4e6), (4e6, 5e6), (5e6, 6e6), (6e6, 7e6)]

for tag, label in (("Self-play/ELO", "ELO"), ("Environment/Episode Length", "episode length")):
    print(f"\n{label} by window")
    print(f"{'run':<14}" + "".join(f"{lo/1e6:.0f}-{hi/1e6:.0f}e6".rjust(11) for lo, hi in WINDOWS))
    for (b, r), c in curves.items():
        s, v = c[tag]
        row = ""
        for lo, hi in WINDOWS:
            m = (s >= lo) & (s < hi)
            row += f"{v[m].mean():>11.1f}" if m.any() else f"{'-':>11}"
        print(f"{r:<14}{row}")

In [ ]:
# fit the trend over each run's final 1e6 steps, alongside the learning rate it had there.
# a flat slope at a healthy learning rate means converged; a flat slope at a dead learning rate
# only means the schedule stopped it
print(f"{'run':<14}{'end step':>10}{'lr at end':>12}{'ELO /1e6':>11}{'ep len /1e6':>13}")
for (b, r), c in curves.items():
    s_elo, v_elo = c["Self-play/ELO"]
    s_len, v_len = c["Environment/Episode Length"]
    s_lr, v_lr = c["Policy/Learning Rate"]
    end = s_elo.max()
    lr_end = v_lr[s_lr >= end - 1e6].mean()
    print(f"{r:<14}{end/1e6:>9.1f}e6{lr_end:>12.2e}"
          f"{slope_per_1e6(s_elo, v_elo, end - 1e6, end):>11.1f}"
          f"{slope_per_1e6(s_len, v_len, end - 1e6, end):>13.2f}")

## Reading it

**260723 had converged by roughly 3e6.** Its ELO is flat or slightly falling over its last million
steps, and its episode length had already levelled off around 2-3e6, while the learning rate was
still about half its peak. Levelling off while the learning rate is still healthy is the part that
makes this convergence rather than annealing. The anneal then locked the policy in rather than
cutting it short.

**260914 had not converged at 4e6, and had still not converged at 7e6.** ELO is still rising and
episode length still falling at the end of every run, at a constant full learning rate with no
anneal at all.

**Neither available comparison is clean.** At 4e6 a converged, annealed 260723 is set against a
mid-flight 260914. At each batch's own final checkpoint, 260723 is converged and annealed while
260914 is neither. The 260914 policies are also still fluctuating around their optimum rather than
settled, which is a candidate explanation for the residual route-efficiency gap in the sims
(`path_ratio` 1.168 against 1.122) on top of any real difference.

**For the next batch.** Keep the schedule kind the same across arms. Mixing a linear anneal in one
arm with a constant rate in another makes every step-matched comparison a comparison of training
regimes as much as of hyperparameters. Either give every arm a linear schedule to its own
`max_steps`, chosen large enough that each converges, or give every arm a constant rate and compare
where all have plateaued. On this evidence a high-gamma arm needs well over 7e6 steps to get there.